# SubCenter ArcFace Hyperparameter Sweep — EfficientNetV2-RW-M

**Goal:** Find optimal SubCenter ArcFace hyperparameters for jaguar re-ID with fewer runs.

EfficientNetV2-RW-M achieved the best mAP (0.787) and CMC@1 (0.797) in backbone experiments.
SubCenter ArcFace was the best-performing loss. This notebook now uses a **Bayesian sweep**
to reduce runtime compared with full-grid search.

## Sweep Design

**Primary sweep** — Bayesian optimization (default 20 runs):
- **Margin (m):** [0.2, 0.3, 0.5, 0.7]
- **Scale (s):** [16, 32, 48, 64]
- **Sub-centers (k):** [1, 2, 3, 4]

**Optional refinement** — small secondary sweep around best m/s/k.

**Fixed Settings:**
- Backbone: `hf-hub:timm/efficientnetv2_rw_m.agc_in1k` (2152-dim, input 320)
- Loss: `subcenter_arcface`
- Epochs: 50, AdamW (lr=1e-4, wd=1e-4), seed 42
- Dataset: `JID_HF_0226_Segmented_Deduplicated_Cached` (1,998 images, 175 identities)
- Evaluation: mAP, CMC@k, identity-balanced mAP, mAP@9+

All runs tracked in W&B: `camera-trap-reidentification`, tags: `hp_sweep`, `subcenter_arcface`

## Setup

In [ ]:
import sys
import copy
import json
import logging
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import fiftyone as fo
import wandb

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.experiments import get_default_config, ExperimentConfig
from jaguars.reidentification.training.train import run_processing as run_training

logger = setup_logger("hp_sweep", level=logging.INFO)
print("Imports OK")

ImportError: cannot import name 'ExperimentConfig' from 'jaguars.reidentification.config' (/sc/home/philipp.kolbe/JID/camera-trap-footage/src/jaguars/reidentification/config.py)

## Base Configuration

In [ ]:
# ── Run metadata ──────────────────────────────────────────────
RUN_BATCH = "hp_sweep_subcenter_arcface"
DATASET_TAG = "dataset:hf_jaguars_camera_trap_0226_segmented_deduplicated"
SOURCE_TAG = "source:fiftyone_local_cache"
LOCAL_FO_DATASET_NAME = "JID_HF_0226_Segmented_Deduplicated_Cached"

# ── Base config (shared across all sweep runs) ──────────────
config = get_default_config()

# W&B
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"

# Dataset (FiftyOne local cache)
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = LOCAL_FO_DATASET_NAME
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"
config.dataset.fo_embeddings_field = None  # recompute from backbone each run
config.dataset.train_split = "train"
config.dataset.val_split = "val"
config.dataset.test_split = "test"

# Backbone: EfficientNetV2-RW-M (best backbone)
config.backbone.name = "hf-hub:timm/efficientnetv2_rw_m.agc_in1k"
config.backbone.pretrained = True
config.backbone.embedding_dim = 2152
config.backbone.input_size = 320

# Loss: SubCenter ArcFace (defaults overridden per experiment)
config.training.loss_name = "subcenter_arcface"
config.training.num_subcenters = 2
config.training.num_epochs = 50
config.training.learning_rate = 1e-4
config.training.weight_decay = 1e-4
config.training.batch_size = 32

# Model head defaults
config.model.embedding_dim = 256
config.model.hidden_dim = 512
config.model.dropout = 0.3

# Verify dataset exists
assert fo.dataset_exists(LOCAL_FO_DATASET_NAME), f"Dataset not found: {LOCAL_FO_DATASET_NAME}"
print(f"✓ Base config ready")
print(f"  Backbone: {config.backbone.name}")
print(f"  Loss: {config.training.loss_name}")
print(f"  Default sub-centers: {config.training.num_subcenters}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Dataset: {LOCAL_FO_DATASET_NAME}")

## Primary Sweep: Bayesian Optimization (Margin × Scale × Sub-centers)

Use W&B Bayesian sweep to optimize the core SubCenter ArcFace hyperparameters:
- **margin (m):** angular margin penalty
- **scale (s):** logit rescaling (includes small-scale option `16`)
- **num_subcenters (k):** sub-centers per class

In [ ]:
# ── Bayesian sweep config ─────────────────────────────────────
SWEEP_RUNS = 20  # cap trials to keep runtime manageable
SWEEP_METRIC = "final_val_map"

sweep_config = {
    "method": "bayes",
    "metric": {"name": SWEEP_METRIC, "goal": "maximize"},
    "parameters": {
        "margin": {"values": [0.2, 0.3, 0.5, 0.7]},
        "scale": {"values": [16, 32, 48, 64]},
        "num_subcenters": {"values": [1, 2, 3, 4]},
    },
}

print("Sweep space:")
print(json.dumps(sweep_config, indent=2))


def run_bayesian_trial() -> None:
    run = wandb.init()
    if run is None:
        raise RuntimeError("wandb.init() failed inside sweep trial")

    try:
        c = copy.deepcopy(config)
        c.model.arcface_margin = float(wandb.config.margin)
        c.model.arcface_scale = float(wandb.config.scale)
        c.training.num_subcenters = int(wandb.config.num_subcenters)

        trial_name = (
            f"m{c.model.arcface_margin}_s{int(c.model.arcface_scale)}_"
            f"k{c.training.num_subcenters}_{run.id}"
        )
        c.wandb.run_name = f"{RUN_BATCH}__{trial_name}"

        # Disable nested wandb init inside training; log manually to active sweep run
        c.wandb.enabled = False

        result = run_training(c)

        summary = {
            "margin": c.model.arcface_margin,
            "scale": c.model.arcface_scale,
            "num_subcenters": c.training.num_subcenters,
            "trial_name": trial_name,
            "best_val_map": result.get("best_val_map"),
            "final_val_map": result.get("final_val_map"),
            "best_val_loss": result.get("best_val_loss"),
            "final_val_loss": result.get("final_val_loss"),
            "num_epochs": result.get("num_epochs"),
        }
        wandb.log(summary)
    finally:
        wandb.finish()

In [ ]:
# ── Launch Bayesian sweep ─────────────────────────────────────
sweep_id = wandb.sweep(
    sweep=sweep_config,
    entity=config.wandb.entity,
    project=config.wandb.project,
 )

print(f"Created sweep: {sweep_id}")
print(f"Running {SWEEP_RUNS} Bayesian trials...")

wandb.agent(
    sweep_id,
    function=run_bayesian_trial,
    count=SWEEP_RUNS,
    entity=config.wandb.entity,
    project=config.wandb.project,
 )

print("✓ Bayesian sweep completed")

In [ ]:
# ── Fetch Bayesian sweep results and build table ──────────────
import pandas as pd

api = wandb.Api()
sweep_path = f"{config.wandb.entity}/{config.wandb.project}/{sweep_id}"
sweep = api.sweep(sweep_path)

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    cfg = dict(run.config)

    margin = summary.get("margin", cfg.get("margin"))
    scale = summary.get("scale", cfg.get("scale"))
    num_subcenters = summary.get("num_subcenters", cfg.get("num_subcenters"))

    if margin is None or scale is None or num_subcenters is None:
        continue

    rows.append({
        "run_id": run.id,
        "name": run.name,
        "margin": float(margin),
        "scale": float(scale),
        "num_subcenters": int(num_subcenters),
        "mAP": summary.get("map", summary.get("final_val_map", float("nan"))),
        "CMC@1": summary.get("cmc@1", float("nan")),
        "CMC@5": summary.get("cmc@5", float("nan")),
        "ib-mAP": summary.get("identity_balanced_map", float("nan")),
        "mAP@9+": summary.get("map_min_total_9", float("nan")),
        "best_val_map": summary.get("best_val_map", float("nan")),
        "final_val_map": summary.get("final_val_map", float("nan")),
    })

df_primary = pd.DataFrame(rows).sort_values("final_val_map", ascending=False)
print("Bayesian Sweep Results (sorted by final_val_map):")
print(df_primary.to_string(index=False, float_format="{:.4f}".format))

if len(df_primary) > 0:
    best = df_primary.iloc[0]
    BEST_MARGIN = best["margin"]
    BEST_SCALE = best["scale"]
    BEST_SUBCENTERS = int(best["num_subcenters"])
    print(
        f"\n★ Best: margin={BEST_MARGIN}, scale={BEST_SCALE}, "
        f"num_subcenters={BEST_SUBCENTERS}, final_val_map={best['final_val_map']:.4f}"
    )
else:
    BEST_MARGIN = 0.5
    BEST_SCALE = 64.0
    BEST_SUBCENTERS = 2
    print("⚠ No results found — using defaults for optional refinement")

### Margin × Scale Heatmap (Best num_subcenters)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

if len(df_primary) > 0:
    # Plot margin × scale heatmaps for best num_subcenters
    df_heat = df_primary[df_primary["num_subcenters"] == BEST_SUBCENTERS]

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))  # ACL full-width

    for ax, metric, title in zip(
        axes,
        ["final_val_map", "best_val_map"],
        ["Final Val mAP", "Best Val mAP"],
    ):
        piv = df_heat.pivot(index="margin", columns="scale", values=metric)
        im = ax.imshow(piv.values, cmap="YlOrRd", aspect="auto")
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels([f"{int(s)}" for s in piv.columns], fontsize=8)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels([f"{m}" for m in piv.index], fontsize=8)
        ax.set_xlabel("Scale (s)", fontsize=9)
        ax.set_ylabel("Margin (m)", fontsize=9)
        ax.set_title(title, fontsize=10)

        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                val = piv.values[i, j]
                if not np.isnan(val):
                    ax.text(
                        j,
                        i,
                        f"{val:.3f}",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="black" if val < np.nanmax(piv.values) * 0.85 else "white",
                    )

        fig.colorbar(im, ax=ax, shrink=0.8)

    plt.suptitle(
        f"Bayesian SubCenter Sweep (k={BEST_SUBCENTERS}) — EfficientNetV2-RW-M",
        fontsize=10,
    )
    plt.tight_layout()

    fig_dir = Path("data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached")
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir / "hp_sweep_heatmap.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(fig_dir / "hp_sweep_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved hp_sweep_heatmap.pdf/png")
else:
    print("No results to plot yet.")

## Optional Refinement: Learning Rate & Embedding Dim

After Bayesian sweep identifies best `margin/scale/num_subcenters`, you can optionally run a small local refinement.
Set `RUN_REFINEMENT = True` in the next cell to enable it.

In [ ]:
# ── Secondary sweep experiments ──────────────────────────────
secondary_experiments = []

# Learning rate sweep (best m/s/k, vary lr)
for lr in [5e-5, 1e-4, 3e-4]:
    c = copy.deepcopy(config)
    c.model.arcface_margin = BEST_MARGIN
    c.model.arcface_scale = float(BEST_SCALE)
    c.training.num_subcenters = BEST_SUBCENTERS
    c.training.learning_rate = lr

    name = f"lr{lr:.0e}"
    c.wandb.run_name = f"{RUN_BATCH}__{name}"
    c.wandb.tags = [
        "hp_sweep",
        "subcenter_arcface",
        "lr_sweep",
        f"lr:{lr}",
        f"margin:{BEST_MARGIN}",
        f"scale:{BEST_SCALE}",
        f"subcenters:{BEST_SUBCENTERS}",
        DATASET_TAG,
        SOURCE_TAG,
        f"run_batch:{RUN_BATCH}",
    ]

    secondary_experiments.append(
        ExperimentConfig(
            name=name,
            description=f"LR={lr} (m={BEST_MARGIN}, s={BEST_SCALE}, k={BEST_SUBCENTERS})",
            base_config=c,
            group="lr_sweep",
        )
    )

# Embedding dimension sweep (best m/s/k, vary emb dim)
for emb_dim in [128, 256, 512]:
    c = copy.deepcopy(config)
    c.model.arcface_margin = BEST_MARGIN
    c.model.arcface_scale = float(BEST_SCALE)
    c.training.num_subcenters = BEST_SUBCENTERS
    c.model.embedding_dim = emb_dim
    c.model.hidden_dim = emb_dim * 2  # keep hidden:emb ratio of 2:1

    name = f"emb{emb_dim}"
    c.wandb.run_name = f"{RUN_BATCH}__{name}"
    c.wandb.tags = [
        "hp_sweep",
        "subcenter_arcface",
        "emb_sweep",
        f"emb_dim:{emb_dim}",
        f"margin:{BEST_MARGIN}",
        f"scale:{BEST_SCALE}",
        f"subcenters:{BEST_SUBCENTERS}",
        DATASET_TAG,
        SOURCE_TAG,
        f"run_batch:{RUN_BATCH}",
    ]

    secondary_experiments.append(
        ExperimentConfig(
            name=name,
            description=f"Embedding dim={emb_dim} (m={BEST_MARGIN}, s={BEST_SCALE}, k={BEST_SUBCENTERS})",
            base_config=c,
            group="emb_sweep",
        )
    )

print(f"✓ {len(secondary_experiments)} secondary experiments:")
for exp in secondary_experiments:
    print(f"  - {exp.name}: {exp.description}")

In [ ]:
# ── Run optional refinement ───────────────────────────────────
RUN_REFINEMENT = False

secondary_results = {}
if RUN_REFINEMENT:
    for i, experiment in enumerate(secondary_experiments, 1):
        logger.info("=" * 60)
        logger.info("[%d/%d] %s", i, len(secondary_experiments), experiment.name)
        logger.info("  %s", experiment.description)
        logger.info("=" * 60)

        try:
            result = run_training(experiment.base_config)
            secondary_results[experiment.name] = result
            logger.info("✓ %s completed", experiment.name)
        except Exception as e:
            logger.error("✗ %s failed: %s", experiment.name, e)
            secondary_results[experiment.name] = {"error": str(e)}

    print(f"\n✓ Optional refinement done: {len(secondary_results)}/{len(secondary_experiments)} completed")
else:
    print("Skipping optional refinement (set RUN_REFINEMENT=True to run)")

## Combined Results

In [ ]:
# ── Aggregate all results ─────────────────────────────────────
all_results = {**primary_results, **secondary_results}

rows = []
for name, res in all_results.items():
    if isinstance(res, dict) and "error" in res:
        continue
    rows.append({
        "name": name,
        "mAP": res.get("map", res.get("final_val_map", float("nan"))),
        "CMC@1": res.get("cmc@1", float("nan")),
        "CMC@5": res.get("cmc@5", float("nan")),
        "ib-mAP": res.get("identity_balanced_map", float("nan")),
        "mAP@9+": res.get("map_min_total_9", float("nan")),
        "best_val_map": res.get("best_val_map", float("nan")),
        "final_val_map": res.get("final_val_map", float("nan")),
    })

df_all = pd.DataFrame(rows).sort_values("final_val_map", ascending=False)
print("All Results (sorted by final_val_map):")
print(df_all.to_string(index=False, float_format="{:.4f}".format))

if len(df_all) > 0:
    best = df_all.iloc[0]
    print(f"\n★ Overall best: {best['name']} — final_val_map={best['final_val_map']:.4f}")

In [ ]:
# ── Save all results to JSON ─────────────────────────────────
output_dir = Path("data/results")
output_dir.mkdir(parents=True, exist_ok=True)

save_data = {
    "sweep_id": sweep_id,
    "sweep_runs": SWEEP_RUNS,
    "primary_sweep": {},
    "secondary_sweep": {},
    "best_primary": (
        {
            "margin": BEST_MARGIN,
            "scale": BEST_SCALE,
            "num_subcenters": BEST_SUBCENTERS,
        }
        if len(df_primary) > 0
        else {}
    ),
}

for name, res in primary_results.items():
    save_data["primary_sweep"][name] = res

for name, res in secondary_results.items():
    save_data["secondary_sweep"][name] = res

with open(output_dir / "hp_sweep_subcenter_arcface.json", "w") as f:
    json.dump(save_data, f, indent=2, default=str)

print(f"Saved to {output_dir / 'hp_sweep_subcenter_arcface.json'}")

In [ ]:
# ── Bar chart: top-N configurations by mAP ───────────────────
if len(df_all) >= 3:
    top_n = min(10, len(df_all))
    df_top = df_all.head(top_n)

    fig, ax = plt.subplots(figsize=(3.3, 3.0))
    y = range(len(df_top))
    bars = ax.barh(y, df_top["mAP"], color="#4C72B0", height=0.6)

    ax.set_yticks(y)
    ax.set_yticklabels(df_top["name"], fontsize=7)
    ax.set_xlabel("mAP", fontsize=9)
    ax.set_title("Top Configurations (SubCenter ArcFace HP Sweep)", fontsize=9)
    ax.invert_yaxis()

    for bar, val in zip(bars, df_top["mAP"]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=7)

    plt.tight_layout()
    fig.savefig(fig_dir / "hp_sweep_top_configs.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(fig_dir / "hp_sweep_top_configs.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved hp_sweep_top_configs.pdf/png")
else:
    print("Not enough results to plot.")